In [1]:
!pip install -q ultralytics fastapi uvicorn pyngrok pillow nest_asyncio python-multipart rasterio mercantile


In [ ]:
!rm -f ngrok ngrok.zip ngrok-stable-linux-amd64.tgz
!wget -O ngrok-stable-linux-amd64.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-stable-linux-amd64.tgz
!chmod +x ngrok
!./ngrok version
!./ngrok authtoken  """""""""""""""""YOUR TOKEN HERE"""""""""""""""""""""""""


In [ ]:
import nest_asyncio, threading, io, time, requests, json
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from pyngrok import ngrok
from PIL import Image
import numpy as np
from ultralytics import YOLO
import mercantile
import uvicorn
import asyncio
nest_asyncio.apply()

# Config
MODEL_PATH = "/content/yolo11x-obb.pt"  # YOLOv11 OBB-trained model
CONF_THRESH = 0.3
IOU_THRESH = 0.5
WMTS_URL = "https://wmts.nlsc.gov.tw/wmts/UAV111A03/default/GoogleMapsCompatible/{z}/{y}/{x}.png"
ZOOM = 20
TILE_SIZE = 256

print("Loading YOLOv11 OBB model...")
model = YOLO(MODEL_PATH)
print("Model loaded:", model.names)

app = FastAPI()

class Rectangle(BaseModel):
    sw_lat: float
    sw_lon: float
    ne_lat: float
    ne_lon: float

def xy_to_lonlat(x, y, tile):
    bounds = mercantile.bounds(tile)
    lon = bounds.west + (x / TILE_SIZE) * (bounds.east - bounds.west)
    lat = bounds.north - (y / TILE_SIZE) * (bounds.north - bounds.south)
    return lon, lat

def download_tile(t):
    url = WMTS_URL.format(z=ZOOM, x=t.x, y=t.y)
    r = requests.get(url)
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    return np.array(img), t

@app.post("/detect")
async def detect(rect: Rectangle):
    tiles = list(mercantile.tiles(rect.sw_lon, rect.sw_lat, rect.ne_lon, rect.ne_lat, ZOOM))
    features = []

    for t in tiles:
        tile_np, tile_obj = download_tile(t)
        results = model.predict(
            tile_np,
            conf=CONF_THRESH,
            iou=IOU_THRESH,
            imgsz=max(tile_np.shape[:2]),
            task="obb",        # OBB mode
            verbose=False
        )[0]

        if hasattr(results, "obb") and len(results.obb) > 0:
            for poly, c, s in zip(results.obb.xyxyxyxy.tolist(), results.obb.cls.tolist(), results.obb.conf.tolist()):
                label = model.names[int(c)].lower()
                if label not in ["small vehicle", "large vehicle"]:
                    continue  # Only detect small and large vehicles

                coords_wgs84 = [xy_to_lonlat(x, y, t) for x, y in poly]

                # Filter detections inside user rectangle
                lats = [lat for lon, lat in coords_wgs84]
                lons = [lon for lon, lat in coords_wgs84]
                if min(lats) >= rect.sw_lat and max(lats) <= rect.ne_lat and \
                   min(lons) >= rect.sw_lon and max(lons) <= rect.ne_lon:
                    features.append({
                        "type": "Feature",
                        "properties": {"label": label, "score": float(s)},
                        "geometry": {
                            "type": "Polygon",
                            "coordinates": [[[lon, lat] for lon, lat in coords_wgs84]]
                        }
                    })

    return {"type": "FeatureCollection", "features": features}

@app.get("/", response_class=HTMLResponse)
async def index():
    html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8"/>
<title>UAV Vehicle Detection (OBB)</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.css"/>
<style>
html, body, #map {{ height: 100%; margin: 0; padding: 0; }}
.control-btn {{ background: white; padding: 6px; margin: 6px; cursor: pointer; border-radius: 4px; box-shadow: 0 1px 4px rgba(0,0,0,0.3); }}
.topright-controls {{ position: absolute; top: 10px; right: 10px; z-index: 1000; }}
</style>
</head>
<body>
<div id="map"></div>
<div class="topright-controls" id="controls"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>
<script>
const map = L.map('map').setView([25.0330, 121.5054], 15);
L.tileLayer('{WMTS_URL}', {{ maxZoom: 20 }}).addTo(map);

const drawnItems = new L.FeatureGroup().addTo(map);
map.addControl(new L.Control.Draw({{
    draw: {{ rectangle: true }},
    edit: {{ featureGroup: drawnItems }}
}}));

map.on(L.Draw.Event.CREATED, e => drawnItems.addLayer(e.layer));

const detectionLayer = L.layerGroup().addTo(map);
const controlsDiv = document.getElementById('controls');

const btnDetect = document.createElement('div');
btnDetect.className = 'control-btn';
btnDetect.innerText = 'Detect Vehicles';
controlsDiv.appendChild(btnDetect);

const btnClear = document.createElement('div');
btnClear.className = 'control-btn';
btnClear.innerText = 'Clear Detections';
controlsDiv.appendChild(btnClear);

const btnDownload = document.createElement('div');
btnDownload.className = 'control-btn';
btnDownload.innerText = 'Download GeoJSON';
controlsDiv.appendChild(btnDownload);

let lastGeoJSON = null;

btnDetect.onclick = async () => {{
    if(drawnItems.getLayers().length===0) {{ alert("Draw a rectangle first!"); return; }}
    const rect = drawnItems.getLayers()[0].getBounds();
    const payload = {{
        sw_lat: rect.getSouthWest().lat,
        sw_lon: rect.getSouthWest().lng,
        ne_lat: rect.getNorthEast().lat,
        ne_lon: rect.getNorthEast().lng
    }};
    detectionLayer.clearLayers();
    const res = await fetch("/detect", {{
        method: "POST",
        headers: {{ "Content-Type": "application/json" }},
        body: JSON.stringify(payload)
    }});
    const data = await res.json();
    lastGeoJSON = data;
    console.log("Detection result:", data);

    data.features.forEach(f => {{
        const coords = f.geometry.coordinates[0].map(c => [c[1], c[0]]);
        const color = f.properties.label === "large vehicle" ? 'blue' : 'red';
        L.polygon(coords, {{color: color, weight:1}})
            .bindTooltip(f.properties.label+" "+f.properties.score.toFixed(2))
            .addTo(detectionLayer);
    }});
}};

btnDownload.onclick = () => {{
    if(!lastGeoJSON) {{ alert("No detection results to download!"); return; }}
    const blob = new Blob([JSON.stringify(lastGeoJSON, null, 2)], {{type: "application/json"}});
    const url = URL.createObjectURL(blob);
    const a = document.createElement("a");
    a.href = url;
    a.download = "detection.geojson";
    a.click();
    URL.revokeObjectURL(url);
}};

btnClear.onclick = () => {{
    detectionLayer.clearLayers();
    lastGeoJSON = null;
}};
</script>
</body>
</html>
"""
    return HTMLResponse(content=html)

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()
